# MATH840 — Lab 6, Challenge 2: Exponential smoothing

**Week 6 | graded assignment 4 of 7 | 8 points | due 23:59 today**

Your series from Week 4, forecast with ETS. **Specify the model by hand first**, from your plots, and
only then let `AutoETS` choose and compare. ETS only: no ARIMA, no regression, no averaging with other
methods.

**From this week accuracy is marked.** Your forecast is scored on the hidden horizon against your own
benchmark: beat it — 3 points, within 10% — 2, worse — 1 with a written explanation.

**Keep the section headings below.** They map one-to-one onto the marking rubric:

| Section | Criteria | Points |
|---|---|---|
| 1 | EDA and visualisation | 1 |
| 2 | Forecast accuracy against your benchmark (scored from your file) | 3 |
| 3 | Justification block | 3 |
| 4 | Code quality and reproducibility | 1 |

Submit `SURNAME_lab06_forecast.csv`, `SURNAME_lab06.pdf` and `SURNAME_lab06.ipynb` to the Week 6
activity on Moodle before 23:59.

## 0. Setup

Given — run it and move on. Nothing in this section is marked.

In [ ]:
!pip install -q statsforecast==2.1.1   # the version the pool was screened with

import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsforecast.models import AutoETS
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120)
print("pandas", pd.__version__)

In [ ]:
SURNAME = ""   # <-- for your submission file names
MY_ID   = ""   # <-- the SAME identifier you have used since Week 1

if not SURNAME.strip() or not MY_ID.strip():
    raise ValueError("Set SURNAME and MY_ID before running the rest.")

### Your series

The same identifier produces the same series as in Week 4. If the code printed below is not the one
from your Lab 4 submission, stop and check `MY_ID`.

In [ ]:
BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data/trackb")


def assign_series(student_id: str, codes: list[str]) -> str:
    digest = hashlib.sha256(student_id.strip().encode("utf-8")).hexdigest()
    return codes[int(digest, 16) % len(codes)]


index = pd.read_csv(f"{BASE}/index.csv")
CODE = assign_series(MY_ID, sorted(index["code"]))
spec = index.set_index("code").loc[CODE]

M    = int(spec["m"])       # seasonal period: 12 monthly, 4 quarterly
H    = int(spec["h"])       # how many steps you are forecasting
FREQ = spec["freq"]         # pandas frequency of the calendar: "MS" or "QS"

s = pd.read_csv(f"{BASE}/{CODE}.csv", parse_dates=["ds"])
y = s["y"].to_numpy(float)
dates = s["ds"]

print(f"series {CODE}: {len(s)} observations, {dates.min().date()} to {dates.max().date()}")
print(f"seasonal period m = {M}, forecast horizon h = {H}, frequency {FREQ}")

### Helpers

Given, and used in exactly this form when your file is read. The benchmark below is computed by the
same rule as the scorer: the best of the four simple methods by RMSE on the last `H` observations. That
is the forecast your submission is compared against — refitted on your whole history.

In [ ]:
def benchmark_forecasts(history: np.ndarray, h: int, m: int) -> dict[str, np.ndarray]:
    # The four simple methods from Week 1.
    T = len(history)
    return {
        "mean":   np.repeat(history.mean(), h),
        "naive":  np.repeat(history[-1], h),
        "snaive": np.array([history[-m + (i % m)] for i in range(h)]),
        "drift":  history[-1] + np.arange(1, h + 1) * (history[-1] - history[0]) / (T - 1),
    }


SCALE = float(np.mean(np.abs(y[M:] - y[:-M])))      # MASE denominator, fixed for this series


def mase(actual, forecast) -> float:
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))) / SCALE)


def rmse(actual, forecast) -> float:
    return float(np.sqrt(np.mean((np.asarray(actual) - np.asarray(forecast)) ** 2)))


def fit_ets(history: np.ndarray, spec: str | None = None, damped: bool | None = None):
    # spec=None lets AutoETS search. "AAdA" is read as "AAA" with damped=True; a trend of "N" cannot be damped.
    if spec is None:
        return AutoETS(season_length=M).fit(y=history)
    spec = str(spec).strip().upper()
    if spec.endswith("DA") or spec.endswith("DM") or spec.endswith("DN"):
        spec, damped = spec[:2] + spec[3:], True
    if len(spec) != 3 or spec[0] not in "AM" or spec[1] not in "NA" or spec[2] not in "ANM":
        raise ValueError(f"spec must be three letters - error (A/M), trend (N/A), season (N/A/M), "
                         f"e.g. 'MAM'; damping goes in `damped` or as 'AAdA'. Got {spec!r}")
    return AutoETS(season_length=M, model=spec,
                   damped=bool(damped) if spec[1] != "N" else None).fit(y=history)


def rolling_mase(spec: str | None, damped: bool | None, history: np.ndarray, h: int,
                 windows: int = 3) -> float:
    # Mean MASE over `windows` origins; the last window is the validation window.
    errors = []
    for k in range(windows):
        end = len(history) - k * h
        fit = fit_ets(history[:end - h], spec, damped)
        errors.append(mase(history[end - h:end], fit.predict(h=h)["mean"]))
    return float(np.mean(errors))


def swing_against_level(values: np.ndarray, ds: pd.Series, m: int) -> pd.DataFrame:
    # One row per complete year: the mean of the year (level) and max minus min (seasonal swing).
    years = pd.DataFrame({"year": ds.dt.year.to_numpy(), "y": values})
    full = years.groupby("year").filter(lambda g: len(g) == m)
    return full.groupby("year")["y"].agg(level="mean", swing=lambda v: v.max() - v.min())


train, valid = y[:-H], y[-H:]
train_dates, valid_dates = dates.iloc[:-H], dates.iloc[-H:]

simple = benchmark_forecasts(train, H, M)
simple_scores = pd.DataFrame({name: {"rmse": rmse(valid, fc), "mase": mase(valid, fc)}
                              for name, fc in simple.items()}).T
BENCHMARK = simple_scores["rmse"].idxmin()
print(simple_scores.round(3).to_string())
print(f"\nyour benchmark: {BENCHMARK} (validation MASE {simple_scores.loc[BENCHMARK, 'mase']:.3f})")

FUTURE = pd.date_range(dates.iloc[-1], periods=H + 1, freq=FREQ)[1:]
print("you are forecasting:", FUTURE[0].date(), "to", FUTURE[-1].date())

## 1. EDA and visualisation

*1 point.* The plots that justify your specification in 3.1 — not more.

- a time plot, read for **trend** (and whether it flattens);
- the **seasonal swing against the level** (`swing_against_level` is given), read for additive or
  multiplicative;
- the validation split on a plot, with your chosen model's forecast and 80% interval over it;
- one sentence per plot. A plot with no reading attached earns nothing.

In [ ]:
# TODO: time plot, swing against level, and read them

**What the plots show.**

→

## 2. The model

*3 points, scored from your submission file* on the hidden horizon against your benchmark:
skill = your MASE / your benchmark's MASE; below 1 — 3 points, up to 1.1 — 2, above — 1.

### 2.1 Your specification, by hand

Fill this in **before** you run `AutoETS`, from Section 1 alone. Error, trend, season — e.g. `"MAM"`,
`"AAA"`, `"MNM"` — and whether the trend is damped.

In [ ]:
MY_SPEC, MY_DAMPED = "", False   # <-- e.g. "MAM", False

mine = fit_ets(train, MY_SPEC, MY_DAMPED)
print("you specified:", mine.model_["method"], "  AICc", round(mine.model_["aicc"], 1),
      "  validation MASE", round(mase(valid, mine.predict(h=H)["mean"]), 3))

### 2.2 `AutoETS`, the shortlist, the rolling origin

Let `AutoETS` choose on `train`, then build a table of specifications — yours, AutoETS's, and the
alternatives your plots leave open — with AICc, validation MASE and `rolling_mase`. Then decide.

The guarantee from Week 1 is about **`AutoETS` fitted on your whole history**: every series in the pool
is one where that beats the benchmark. Choosing a different model is legitimate, and it is what 3.2
asks you to argue — but then the evidence for it is yours.

In [ ]:
# TODO: AutoETS on train, then the table of candidates.
#
#   auto = fit_ets(train)
#   print(auto.model_["method"])
#
#   rows.append({"spec": ..., "model": fit.model_["method"], "AICc": fit.model_["aicc"],
#                "valid MASE": mase(valid, fit.predict(h=H)["mean"]),
#                "rolling MASE": rolling_mase(spec, damped, y, H)})

**Your table, and the model you chose.**

→

### 2.3 Residuals and interval on the validation window

The **innovation** residuals of your chosen model fitted on `train` (`fit.model_["residuals"]`): ACF and
Ljung-Box. Then the 80% interval over the validation window: how many of the `H` points does it cover?

In [ ]:
# TODO: residual diagnostics and interval coverage of your chosen model

### 2.4 The submission

Refit your chosen specification on the **whole** history `y` and forecast `H` steps. The interval comes
from the model: `predict(h=H, level=[80])`.

If you chose `AutoETS`, submit `fit_ets(y)` — the search rerun on all of `y` — and print which model it
picked. On half the pool that is a different model from the one it chose on `train`; the guarantee is
about the rerun. Say in 3.2 whether the letters changed.

In [ ]:
# TODO: your final forecast, fitted on all of y.
#
#   final = fit_ets(y, FINAL_SPEC, FINAL_DAMPED)
#   fc = final.predict(h=H, level=[80])
YHAT = None          # np.array of length H  (fc["mean"])
YHAT_LO_80 = None    # np.array of length H  (fc["lo-80"])
YHAT_HI_80 = None    # np.array of length H  (fc["hi-80"])

for name, value in [("YHAT", YHAT), ("YHAT_LO_80", YHAT_LO_80), ("YHAT_HI_80", YHAT_HI_80)]:
    if value is None or len(np.ravel(value)) != H:
        raise ValueError(f"{name} must be an array of length H = {H}")

## 3. Justification block

*3 points, and the part a human reads closely.* Roughly a paragraph each. Numbers from your own cells,
not adjectives.

**3.1 Your specification, before the machine** *(1)* — each letter of `MY_SPEC`, and damping, tied to a
plot from Section 1. What would have made you choose differently?

→

**3.2 What `AutoETS` chose, and what you submitted** *(1)* — where AutoETS agrees or disagrees with you,
and why you think it does; your shortlist with AICc and rolling MASE; the model you submitted and why
that one. If it is not AutoETS's choice, say what evidence made you leave it.

→

**3.3 What is left, and what you expect** *(1)* — the residuals read (structure left? what does it damage
— the forecast or the interval?), the interval's coverage on validation, and your expected skill against
the benchmark on the hidden horizon. If you expect to lose, say why.

→

## 4. Code quality and reproducibility

*1 point.* Before you export:

- Restart the kernel and run everything top to bottom. It must finish without errors.
- The submission file is written by the cell below, not edited by hand.
- No leakage: candidates are compared on `train`/`valid` and `rolling_mase`; only the final model sees
  all of `y`.
- Every number in your text comes from a cell.

## 5. Submit

Run both cells, then upload **three files** to the Week 6 activity on Moodle before 23:59:

- `SURNAME_lab06_forecast.csv` — written below;
- `SURNAME_lab06.pdf` — `File → Print → Save as PDF`;
- `SURNAME_lab06.ipynb` — `File → Download → Download .ipynb`.

In [ ]:
submission = pd.DataFrame({
    "student_id": MY_ID,
    "code": CODE,
    "ds": FUTURE.strftime("%Y-%m-%d"),
    "yhat": np.ravel(YHAT).astype(float),
    "yhat_lo_80": np.ravel(YHAT_LO_80).astype(float),
    "yhat_hi_80": np.ravel(YHAT_HI_80).astype(float),
})

name = f"{SURNAME.strip().upper()}_lab06_forecast.csv"
submission.to_csv(name, index=False)
print(submission.to_string(index=False))

In [ ]:
# The same checks used when your file is read. If this cell passes, your file is fine.
check = pd.read_csv(name, parse_dates=["ds"])
assert len(check) == H, f"expected {H} rows, got {len(check)}"
assert list(check.columns) == ["student_id", "code", "ds", "yhat", "yhat_lo_80", "yhat_hi_80"]
assert (check["ds"].to_numpy() == FUTURE.to_numpy()).all(), "the dates are not the H dates asked for"
assert check[["yhat", "yhat_lo_80", "yhat_hi_80"]].notna().all().all(), "no NaN allowed"
assert (check["yhat_lo_80"] <= check["yhat"]).all() and (check["yhat"] <= check["yhat_hi_80"]).all()
print("format OK -", name, "is ready to upload")

try:
    from google.colab import files
    files.download(name)
except ImportError:
    print(f"Not in Colab - {name} is saved next to this notebook.")